# Criação das tabelas do Data Warehouse

In [1]:
#Imports

import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text
import re
import numpy as np
from typing import Any


username = "root"
password = "pass"  
host = "localhost"
port = 3306
database = "DSIA2"

engine = create_engine(f'mysql+mysqlconnector://{username}:{password}@{host}:{port}/{database}')

In [2]:
with engine.begin() as conn:
    conn.execute(text("""
        SET FOREIGN_KEY_CHECKS = 0
    """))

    conn.execute(text("""
        DROP TABLE IF EXISTS 
            detalhe_compra,
            compra,
            venda,
            inventario,
            preco,
            produto_inventario, 
            produto,
            fornecedor,
            loja,
            calendario
    """))

    conn.execute(text("""
        SET FOREIGN_KEY_CHECKS = 1
    """))


In [3]:
#Criar tabela Fornecedor

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS fornecedor"))
    conn.execute(text("""
        
        CREATE TABLE IF NOT EXISTS fornecedor (
            ID_Fornecedor INT NOT NULL,
            Nome VARCHAR(255) NOT NULL,
            PRIMARY KEY (ID_Fornecedor)
        )
    """))

In [4]:
#Criar tabela Produto

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS produto"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS produto (
            ID_Produto INT NOT NULL AUTO_INCREMENT,
            Descricao VARCHAR(255) NOT NULL,
            Marca VARCHAR(255) NOT NULL,
            Tamanho VARCHAR(50) NOT NULL,
            Volume DECIMAL(10,2) NOT NULL DEFAULT 0,
            Unidades_Pack INT NOT NULL DEFAULT 1,
            Tipo_Pack VARCHAR(50) NOT NULL DEFAULT 'UNIT',
            Classificacao INT NOT NULL DEFAULT 0,
            PRIMARY KEY (ID_Produto),
            UNIQUE KEY UK_produto (Marca, Descricao, Tamanho, Volume, Unidades_Pack, Tipo_Pack)
        )
    """))

Produto_inventário é uma tabela auxiliar para mapear os produtos que têm InventoryID sem colocar InventoryID na dimensão produto

In [5]:
#Criar tabela auxiliar Produto-Inventario

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS produto_inventario"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS produto_inventario (
            ID_Inventario VARCHAR(255) NOT NULL,
            ID_Produto INT NOT NULL,
            PRIMARY KEY (ID_Inventario),
            FOREIGN KEY (ID_Produto) REFERENCES produto(ID_Produto)
        )
    """))

In [6]:
#Criar tabela Calendario

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS calendario"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS calendario (
            ID_Calendario INT NOT NULL,
            Data DATE NOT NULL,
            Dia INT NOT NULL,
            Mes INT NOT NULL,
            Semestre INT NOT NULL,
            Ano INT NOT NULL,
            PRIMARY KEY (ID_Calendario)
        )
    """))

In [7]:
#Criar tabela Loja

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS loja"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS loja (
            ID_Loja INT NOT NULL,
            N_Loja VARCHAR(255) NOT NULL,
            Cidade VARCHAR(255) NOT NULL,
            PRIMARY KEY (ID_Loja)
        )
    """))

In [8]:
#Criar tabela Inventario

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS inventario"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS inventario (
            ID_Calendario INT NOT NULL,
            ID_Produto INT NOT NULL,
            ID_Loja INT NOT NULL,
            
            Existencias INT NOT NULL,
            ID_Inventario VARCHAR(255) NOT NULL,
            
            PRIMARY KEY (ID_Produto, ID_Loja, ID_Calendario),
            FOREIGN KEY (ID_Produto) REFERENCES produto(ID_Produto),
            FOREIGN KEY (ID_Loja) REFERENCES loja(ID_Loja),
            FOREIGN KEY (ID_Calendario) REFERENCES calendario(ID_Calendario)
        )
    """))

In [9]:
#Criar tabela Venda

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS venda"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS venda (
            ID_Calendario INT NOT NULL,
            ID_Loja INT NOT NULL,
            ID_Produto INT NOT NULL,
                      
            ID_Inventario VARCHAR(255) NOT NULL,
                      
            Preco_Produto_Venda FLOAT NOT NULL,
            Valor_Venda FLOAT NOT NULL,
            Quantidade_Vendida INT NOT NULL,
            Imposto_Sobre_Venda FLOAT NOT NULL,
            
            PRIMARY KEY (ID_Calendario, ID_Loja, ID_Produto),
            FOREIGN KEY (ID_Calendario) REFERENCES calendario(ID_Calendario),
            FOREIGN KEY (ID_Loja) REFERENCES loja(ID_Loja),
            FOREIGN KEY (ID_Produto) REFERENCES produto(ID_Produto)
        )
    """))

In [10]:
#Criar tabela Preco

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS preco"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS preco (
            ID_Calendario INT NOT NULL,
            ID_Produto INT NOT NULL,
            ID_Fornecedor INT NOT NULL,
            
            Preco_Produto FLOAT NOT NULL,
            
            PRIMARY KEY (ID_Calendario, ID_Produto, ID_Fornecedor),
            FOREIGN KEY (ID_Calendario) REFERENCES calendario(ID_Calendario),
            FOREIGN KEY (ID_Produto) REFERENCES produto(ID_Produto),
            FOREIGN KEY (ID_Fornecedor) REFERENCES fornecedor(ID_Fornecedor)
        )
    """))

In [11]:
#Criar tabela Detalhe_Compra

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS detalhe_compra"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS detalhe_compra (
            ID_Detalhe_Compra BIGINT NOT NULL AUTO_INCREMENT,
            ID_Calendario INT NOT NULL,
            ID_Fornecedor INT NOT NULL,
            ID_Produto INT NOT NULL,
            ID_Loja INT NOT NULL,
                      
            N_Fatura INT NOT NULL,
            ID_Inventario VARCHAR(255) NOT NULL,
            
            Quantidade_Comprada INT NOT NULL,
            Preco_Compra_Produto FLOAT NOT NULL,
            Valor_Linha_Compra FLOAT NOT NULL,
            
            PRIMARY KEY (ID_Detalhe_Compra),
            FOREIGN KEY (ID_Calendario) REFERENCES calendario(ID_Calendario),
            FOREIGN KEY (ID_Fornecedor) REFERENCES fornecedor(ID_Fornecedor),
            FOREIGN KEY (ID_Produto) REFERENCES produto(ID_Produto),
            FOREIGN KEY (ID_Loja) REFERENCES loja(ID_Loja)
        )
    """))

In [3]:
#Criar tabela Compra

with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS compra"))
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS compra (
            ID_Calendario INT NOT NULL,
            ID_Fornecedor INT NOT NULL,
            ID_Loja INT NOT NULL,
                      
            N_Fatura INT NOT NULL,
            
            Frete FLOAT NOT NULL,
                      
            PRIMARY KEY (ID_Calendario, ID_Fornecedor, ID_Loja, N_Fatura),
            FOREIGN KEY (ID_Calendario) REFERENCES calendario(ID_Calendario),
            FOREIGN KEY (ID_Fornecedor) REFERENCES fornecedor(ID_Fornecedor),
            FOREIGN KEY (ID_Loja) REFERENCES loja(ID_Loja)
        )
    """))

Inserção dos Dados

# CALENDÁRIO

In [13]:
df_beg_inventory = pd.read_sql(text("SELECT * FROM beg_inventory"), con=engine)
df_end_inventory = pd.read_sql(text("SELECT * FROM end_inventory"), con=engine)
df_purchases = pd.read_sql(text("SELECT * FROM purchases"), con=engine)
df_invoice_purchases = pd.read_sql(text("SELECT * FROM invoice_purchases"), con=engine)
df_sales = pd.read_sql(text("SELECT * FROM sales"), con=engine)

datas = pd.concat([
    pd.to_datetime(df_invoice_purchases["InvoiceDate"], errors="coerce"),
    pd.to_datetime(df_invoice_purchases["PODate"], errors="coerce"),
    pd.to_datetime(df_invoice_purchases["PayDate"], errors="coerce"),
    pd.to_datetime(df_beg_inventory["startDate"], errors="coerce"),
    pd.to_datetime(df_end_inventory["endDate"], errors="coerce"),
    pd.to_datetime(df_sales["SalesDate"], errors="coerce"),
    pd.to_datetime(df_purchases["PODate"], errors="coerce"),
    pd.to_datetime(df_purchases["PayDate"], errors="coerce"),
    pd.to_datetime(df_purchases["ReceivingDate"], errors="coerce"),
    pd.to_datetime(df_purchases["InvoiceDate"], errors="coerce"),
], ignore_index=True)

datas_unicas = (datas.dropna().drop_duplicates().sort_values().reset_index(drop=True))

df_existente = pd.read_sql("SELECT ID_Calendario, Data FROM calendario", engine)
df_existente["Data"] = pd.to_datetime(df_existente["Data"], errors="coerce")

datas_existentes = set(df_existente["Data"].dropna())
datas_novas = datas_unicas[~datas_unicas.isin(datas_existentes)].reset_index(drop=True)

print(f"Datas novas a inserir: {len(datas_novas)}")

if len(datas_novas) > 0:
    max_id = int(df_existente["ID_Calendario"].max()) if not df_existente.empty else 0

    df_novas_datas = pd.DataFrame({
        "ID_Calendario": range(max_id + 1, max_id + 1 + len(datas_novas)),
        "Data": datas_novas.values,
        "Dia": datas_novas.dt.day.values,
        "Mes": datas_novas.dt.month.values,
        "Semestre": ((datas_novas.dt.month - 1) // 6 + 1).values,
        "Ano": datas_novas.dt.year.values
    })

    df_novas_datas.to_sql("calendario", con=engine, if_exists="append", index=False)

    print(f"Inseridas {len(df_novas_datas)} novas linhas em calendario.")
else:
    print("Não há novas datas para inserir.")

Datas novas a inserir: 427
Inseridas 427 novas linhas em calendario.


# FORNECEDOR

In [14]:
df_invoice_purchases = pd.read_sql(text("SELECT VendorNumber AS ID_Fornecedor, VendorName AS Nome FROM invoice_purchases"), con=engine)
df_purchases = pd.read_sql(text("SELECT VendorNumber AS ID_Fornecedor, VendorName AS Nome FROM purchases"), con=engine)
df_purchase_prices = pd.read_sql(text("SELECT VendorNumber AS ID_Fornecedor, VendorName AS Nome FROM purchase_prices"), con=engine)
df_sales = pd.read_sql(text("SELECT VendorNo AS ID_Fornecedor, VendorName AS Nome FROM sales"), con=engine)

df_fornecedor_original = pd.concat([df_invoice_purchases, df_purchases, df_purchase_prices, df_sales], ignore_index=True)

df_fornecedor = df_fornecedor_original.copy()
df_fornecedor["ID_Fornecedor"] = pd.to_numeric(df_fornecedor["ID_Fornecedor"], errors="coerce")
df_fornecedor["Nome"] = (df_fornecedor["Nome"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True))

df_fornecedor = df_fornecedor.dropna(subset=["ID_Fornecedor", "Nome"])
df_fornecedor = df_fornecedor[df_fornecedor["Nome"] != ""]
df_fornecedor["ID_Fornecedor"] = df_fornecedor["ID_Fornecedor"].astype(int)

df_fornecedor = (df_fornecedor.drop_duplicates(subset=["ID_Fornecedor"], keep="first").reset_index(drop=True)
)

df_existente = pd.read_sql("SELECT ID_Fornecedor, Nome FROM fornecedor", engine)
df_existente["ID_Fornecedor"] = pd.to_numeric(df_existente["ID_Fornecedor"], errors="coerce")

fornecedor_existente = set(df_existente["ID_Fornecedor"].dropna())
fornecedor_novo = df_fornecedor[~df_fornecedor["ID_Fornecedor"].isin(fornecedor_existente)].reset_index(drop=True)

print(f"Fornecedores novos a inserir: {len(fornecedor_novo)}")

if len(fornecedor_novo) > 0:
    fornecedor_novo.to_sql("fornecedor", con=engine, if_exists="append", index=False)
    print(f"Inseridos {len(fornecedor_novo)} novos fornecedores.")

else:
    print("Não há novos fornecedores para inserir.")

Fornecedores novos a inserir: 132
Inseridos 132 novos fornecedores.


# LOJA

In [15]:
df_beg_inventory = pd.read_sql(text("SELECT Store AS N_Loja, City AS Cidade FROM beg_inventory"), con=engine)
df_end_inventory = pd.read_sql(text("SELECT Store AS N_Loja, City AS Cidade FROM end_inventory"), con=engine)
df_sales = pd.read_sql(text("SELECT Store AS N_Loja FROM sales"), con=engine)
df_purchases = pd.read_sql(text("SELECT Store AS N_Loja FROM purchases"), con=engine)

df_loja_original = pd.concat([df_beg_inventory, df_end_inventory, df_sales, df_purchases], ignore_index=True)

df_loja_original["N_Loja"] = (df_loja_original["N_Loja"].astype("string").str.strip())
df_loja_original["Cidade"] = (df_loja_original["Cidade"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True))


df_loja_original["StoreCodigo"] = (df_loja_original["N_Loja"].str.split("_", n=1).str[0].str.strip())

lojas_unicas = (df_loja_original[df_loja_original["StoreCodigo"].notna()].sort_values(by=["Cidade"], na_position="last").drop_duplicates(subset=["StoreCodigo"], keep="first")[["StoreCodigo", "Cidade"]].reset_index(drop=True))

df_existente = pd.read_sql("SELECT ID_Loja, N_Loja, Cidade FROM loja", engine)
df_existente["ID_Loja"] = pd.to_numeric(df_existente["ID_Loja"], errors="coerce")
df_existente["StoreCodigo"] = (df_existente["N_Loja"].astype("string").str.strip().str.split("_", n=1).str[0].str.strip())

loja_existente = set(df_existente["StoreCodigo"].dropna())
loja_nova = lojas_unicas[~lojas_unicas["StoreCodigo"].isin(loja_existente)].reset_index(drop=True)

print(f"Lojas novas a inserir: {len(loja_nova)}")

if len(loja_nova) > 0:
    max_id = int(df_existente["ID_Loja"].max()) if not df_existente.empty else 0
    loja_nova["ID_Loja"] = range(max_id + 1, max_id + 1 + len(loja_nova))

    loja_nova["Cidade"] = (
        loja_nova["Cidade"]
        .fillna("SEM_CIDADE")
        .replace("<NA>", "SEM_CIDADE")
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
)
    loja_nova["N_Loja"] = loja_nova["StoreCodigo"] + "_" + loja_nova["Cidade"]

    loja_nova = loja_nova[["ID_Loja", "N_Loja", "Cidade"]]
    loja_nova.to_sql("loja", con=engine, if_exists="append", index=False)
    print(f"Inseridas {len(loja_nova)} novas lojas.")
else:
    print("Não há novas lojas para inserir.")

Lojas novas a inserir: 80
Inseridas 80 novas lojas.


# PRODUTO

In [16]:

def primeiro_nao_nulo(serie):
    valores_nao_nulos = serie.dropna()
    return valores_nao_nulos.iloc[0] if len(valores_nao_nulos) else pd.NA

def normaliza_txt(serie):
    return (
        serie.astype("string")
        .fillna("")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

def normaliza_volume(serie):
    return pd.to_numeric(serie, errors="coerce").fillna(0).round(2)

def volume_para_chave(serie):
    return normaliza_volume(serie).map(lambda valor: f"{valor:.2f}")

def tamanho_para_volume_ml(valor):
    if pd.isna(valor):
        return pd.NA
    texto_tamanho = str(valor).strip().lower().replace(",", ".")
    if texto_tamanho in {"", "unknown"}:
        return pd.NA

    ml_por_oz = 29.5735
    ml_por_gal = 3785.411784

    if texto_tamanho in {"liter"}:
     return 1000

    if texto_tamanho in {"25", "25.0"}:
     return 25
    
    match_volume = re.search(r"(\d+(?:\.\d+)?)\s*(mls?|l|oz|gal|liter|litro)\b", texto_tamanho)
    if not match_volume:
        return pd.NA

    quantidade = float(match_volume.group(1))
    unidade = match_volume.group(2)
    if unidade in {"ml", "mls"}:
        return int(round(quantidade))
    if unidade in {"l", "liter"}:
        return int(round(quantidade * 1000))
    if unidade == "oz":
        return int(round(quantidade * ml_por_oz))
    if unidade == "gal":
        return int(round(quantidade * ml_por_gal))
    return pd.NA

def normaliza_descricao_produto(serie):
    s = normaliza_txt(serie).str.upper()
    s = s.str.replace(r"\bW\s*/\s*", " WITH ", regex=True)
    s = s.str.replace(r"\bWITH\s*(\d)", r"WITH \1", regex=True)
    s = s.str.replace(r"(\d)\s*MLS\b", r"\1ML", regex=True)
    s = s.str.replace(r"\bPAK\b", "PACK", regex=True)
    s = s.str.replace(r"\bPK\b", "PACK", regex=True)
    s = s.str.replace(r"[^A-Z0-9]+", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def unidades_por_pack(valor):
    if pd.isna(valor):
        return 1
    texto = str(valor).strip().lower().replace(",", ".")
    match_barra = re.search(r"^\s*(\d+)\s*/\s*\d", texto)
    if match_barra:
        return int(match_barra.group(1))
    match_pack = re.search(r"\b(\d+)\s*(?:pk|pack|pak)\b", texto)
    if match_pack:
        return int(match_pack.group(1))
    match_bonus = re.search(r"\+\s*(\d+)\s*/?", texto)
    if match_bonus:
        return 1 + int(match_bonus.group(1))
    return 1

def tipo_pack(valor):
    if pd.isna(valor):
        return "UNIT"
    texto = str(valor).strip().lower()
    if "+" in texto:
        return "BONUS"
    if re.search(r"^\s*\d+\s*/\s*\d", texto):
        return "PACK"
    if re.search(r"\b\d+\s*(?:pk|pack|pak)\b", texto):
        return "PACK"
    return "UNIT"

def volume_unitario_produto(linha):
    volume_origem = linha.get("Volume_Origem")
    volume_tamanho = linha.get("Volume_Tamanho")
    unidades = linha.get("Unidades_Pack")

    volume_origem_valido = pd.notna(volume_origem) and float(volume_origem) > 0
    volume_tamanho_valido = pd.notna(volume_tamanho) and float(volume_tamanho) > 0
    unidades = int(unidades) if pd.notna(unidades) and int(unidades) > 0 else 1

    if not volume_origem_valido:
        return volume_tamanho if volume_tamanho_valido else pd.NA
    if not volume_tamanho_valido:
        return volume_origem

    volume_origem = float(volume_origem)
    volume_tamanho = float(volume_tamanho)
    if unidades > 1:
        total_estimado = volume_tamanho * unidades
        if abs(volume_origem - total_estimado) <= max(2, total_estimado * 0.03):
            return volume_tamanho
    if abs(volume_origem - volume_tamanho) <= max(1, volume_tamanho * 0.005):
        return volume_tamanho
    return volume_origem

df_beg_inventory = pd.read_sql(text("""
    SELECT InventoryId AS ID_Inventario, Brand AS Marca, Description AS Descricao, Size AS Tamanho
    FROM beg_inventory
"""), con=engine).assign(Classificacao=np.nan, Volume=np.nan, Fonte="beg_inventory")

df_end_inventory = pd.read_sql(text("""
    SELECT InventoryId AS ID_Inventario, Brand AS Marca, Description AS Descricao, Size AS Tamanho
    FROM end_inventory
"""), con=engine).assign(Classificacao=np.nan, Volume=np.nan, Fonte="end_inventory")

df_purchases = pd.read_sql(text("""
    SELECT InventoryId AS ID_Inventario, Brand AS Marca, Description AS Descricao, Size AS Tamanho, Classification AS Classificacao
    FROM purchases
"""), con=engine).assign(Volume=np.nan, Fonte="purchases")

df_sales = pd.read_sql(text("""
    SELECT InventoryId AS ID_Inventario, Brand AS Marca, Description AS Descricao, Size AS Tamanho, Volume AS Volume
    FROM sales
"""), con=engine).assign(Classificacao=np.nan, Fonte="sales")

df_purchase_prices = pd.read_sql(text("""
    SELECT NULL AS ID_Inventario, Brand AS Marca, Description AS Descricao, Size AS Tamanho, Volume AS Volume
    FROM purchase_prices
"""), con=engine).assign(Classificacao=np.nan, Fonte="purchase_prices")

brands_usados = pd.concat([
    df_beg_inventory["Marca"],
    df_end_inventory["Marca"],
    df_purchases["Marca"],
    df_sales["Marca"],
], ignore_index=True)
brands_usados = normaliza_txt(brands_usados).str.upper()
brands_usados = brands_usados[brands_usados.ne("")].unique()

df_produto_original = pd.concat([df_purchase_prices, df_sales, df_purchases, df_beg_inventory, df_end_inventory], ignore_index=True)

for c in ["Descricao", "Marca", "Tamanho"]:
    df_produto_original[c] = normaliza_txt(df_produto_original[c])

df_produto_original["Marca_Norm"] = df_produto_original["Marca"].str.upper()
df_produto_original = df_produto_original[df_produto_original["Marca_Norm"].isin(brands_usados)].copy()

produto_completo = (df_produto_original["Marca"].ne("") &df_produto_original["Descricao"].ne("") &df_produto_original["Tamanho"].ne(""))

df_produto_original = df_produto_original[produto_completo].copy()

df_produto_original["ID_Inventario"] = normaliza_txt(df_produto_original["ID_Inventario"]).str.upper()
df_produto_original["Classificacao"] = pd.to_numeric(df_produto_original["Classificacao"], errors="coerce")
df_produto_original["Descricao_Norm"] = normaliza_descricao_produto(df_produto_original["Descricao"])
df_produto_original["Tamanho_Norm"] = normaliza_txt(df_produto_original["Tamanho"]).str.upper()
df_produto_original["Unidades_Pack"] = df_produto_original["Tamanho"].apply(unidades_por_pack).astype(int)
df_produto_original["Tipo_Pack"] = df_produto_original["Tamanho"].apply(tipo_pack)
df_produto_original["Volume_Origem"] = pd.to_numeric(df_produto_original["Volume"], errors="coerce")
df_produto_original["Volume_Tamanho"] = df_produto_original["Tamanho"].apply(tamanho_para_volume_ml)
df_produto_original["Volume"] = df_produto_original.apply(volume_unitario_produto, axis=1)
df_produto_original["Volume"] = normaliza_volume(df_produto_original["Volume"])

df_produto_original["_nk"] = (
    df_produto_original["Marca_Norm"] + "|" +
    df_produto_original["Descricao_Norm"] + "|" +
    df_produto_original["Tamanho_Norm"] + "|" +
    volume_para_chave(df_produto_original["Volume"]) + "|" +
    df_produto_original["Unidades_Pack"].astype(str) + "|" +
    df_produto_original["Tipo_Pack"]
)

fontes_operacionais = {"sales", "purchases", "beg_inventory", "end_inventory"}
chaves_usadas = df_produto_original[df_produto_original["Fonte"].isin(fontes_operacionais)]["_nk"].unique()
df_produto_original = df_produto_original[df_produto_original["_nk"].isin(chaves_usadas)].copy()

prioridade_fonte = {
    "purchase_prices": 1,
    "purchases": 2,
    "sales": 3,
    "end_inventory": 4,
    "beg_inventory": 5,
}

df_produto_original["Prioridade_Fonte"] = df_produto_original["Fonte"].map(prioridade_fonte).fillna(99)
df_produto_original = df_produto_original.sort_values(["_nk", "Prioridade_Fonte"])

df_produtos_preparados = (
    df_produto_original
    .dropna(subset=["Marca", "Descricao", "Tamanho"])
    .groupby("_nk", as_index=False)
    .agg({
        "Descricao": primeiro_nao_nulo,
        "Marca": primeiro_nao_nulo,
        "Tamanho": primeiro_nao_nulo,
        "Volume": primeiro_nao_nulo,
        "Unidades_Pack": primeiro_nao_nulo,
        "Tipo_Pack": primeiro_nao_nulo,
        "Classificacao": "max",
    })
)

df_produtos_preparados["Volume"] = normaliza_volume(df_produtos_preparados["Volume"])
df_produtos_preparados["Unidades_Pack"] = pd.to_numeric(df_produtos_preparados["Unidades_Pack"], errors="coerce").fillna(1).astype(int)
df_produtos_preparados["Classificacao"] = (
    pd.to_numeric(df_produtos_preparados["Classificacao"], errors="coerce")
    .fillna(0)
    .astype(int)
)

cols = ["Descricao", "Marca", "Tamanho", "Volume", "Unidades_Pack", "Tipo_Pack", "Classificacao"]

dados_produto = df_produtos_preparados[cols].copy()

dados_produto = dados_produto.astype(object).where(pd.notna(dados_produto), None)

registos: list[dict[str, Any]] = [
    {str(chave): valor for chave, valor in linha.items()}
    for linha in dados_produto.to_dict(orient="records")
]

with engine.begin() as conn:
    conn.execute(text("""
        INSERT INTO produto (Descricao, Marca, Tamanho, Volume, Unidades_Pack, Tipo_Pack, Classificacao)
        VALUES (:Descricao, :Marca, :Tamanho, :Volume, :Unidades_Pack, :Tipo_Pack, :Classificacao)
        ON DUPLICATE KEY UPDATE
            Descricao = VALUES(Descricao),
            Tamanho = VALUES(Tamanho),
            Volume = COALESCE(VALUES(Volume), produto.Volume),
            Unidades_Pack = VALUES(Unidades_Pack),
            Tipo_Pack = VALUES(Tipo_Pack),
            Classificacao = COALESCE(VALUES(Classificacao), produto.Classificacao)
    """), registos)
    conn.execute(text("UPDATE produto SET Classificacao = 0 WHERE Classificacao IS NULL"))

print(f"Produtos inseridos: {len(df_produtos_preparados)}")

Produtos inseridos: 11788


# Tabela auxiliar - Produto-Inventário

In [17]:

def normaliza_txt(serie):
    return (
        serie.astype("string")
        .fillna("")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

def normaliza_volume(serie):
    return pd.to_numeric(serie, errors="coerce").fillna(0).round(2)

def volume_para_chave(serie):
    return normaliza_volume(serie).map(lambda valor: f"{valor:.2f}")
 
df_prod_dim = pd.read_sql(
    text("SELECT ID_Produto, Marca, Descricao, Tamanho, Volume, Unidades_Pack, Tipo_Pack FROM produto"),
    con=engine,
 )
 
for c in ["Marca", "Descricao", "Tamanho", "Tipo_Pack"]:
    df_prod_dim[c] = normaliza_txt(df_prod_dim[c])
df_prod_dim["Descricao_Norm"] = normaliza_descricao_produto(df_prod_dim["Descricao"])
df_prod_dim["Tamanho_Norm"] = normaliza_txt(df_prod_dim["Tamanho"]).str.upper()
df_prod_dim["Volume"] = normaliza_volume(df_prod_dim["Volume"])
df_prod_dim["Unidades_Pack"] = pd.to_numeric(df_prod_dim["Unidades_Pack"], errors="coerce").fillna(1).astype(int)
 
df_prod_dim["_nk"] = (
    df_prod_dim["Marca"].str.upper() + "|" +
    df_prod_dim["Descricao_Norm"].str.upper() + "|" +
    df_prod_dim["Tamanho_Norm"].str.upper() + "|" +
    volume_para_chave(df_prod_dim["Volume"]) + "|" +
    df_prod_dim["Unidades_Pack"].astype(str) + "|" +
    df_prod_dim["Tipo_Pack"].str.upper()
)
 
df_prod_nk = (df_prod_dim.sort_values("ID_Produto").drop_duplicates(subset=["_nk"], keep="first")[["ID_Produto", "_nk"]].reset_index(drop=True))
 
df_map_src = pd.concat([
    pd.read_sql(text("""
        SELECT InventoryId AS ID_Inventario, Brand AS Marca, Description AS Descricao, Size AS Tamanho, Volume AS Volume
        FROM sales
    """), con=engine).assign(Fonte="sales"),
    pd.read_sql(text("""
        SELECT InventoryId AS ID_Inventario, Brand AS Marca, Description AS Descricao, Size AS Tamanho, NULL AS Volume
        FROM purchases
    """), con=engine).assign(Fonte="purchases"),
    pd.read_sql(text("""
        SELECT InventoryId AS ID_Inventario, Brand AS Marca, Description AS Descricao, Size AS Tamanho, NULL AS Volume
        FROM beg_inventory
    """), con=engine).assign(Fonte="beg_inventory"),
    pd.read_sql(text("""
        SELECT InventoryId AS ID_Inventario, Brand AS Marca, Description AS Descricao, Size AS Tamanho, NULL AS Volume
        FROM end_inventory
    """), con=engine).assign(Fonte="end_inventory"),
 ], ignore_index=True)
 
df_map_src["ID_Inventario"] = normaliza_txt(df_map_src["ID_Inventario"]).str.upper()
for c in ["Marca", "Descricao", "Tamanho"]:
    df_map_src[c] = normaliza_txt(df_map_src[c])
df_map_src["Descricao_Norm"] = normaliza_descricao_produto(df_map_src["Descricao"])
df_map_src["Tamanho_Norm"] = normaliza_txt(df_map_src["Tamanho"]).str.upper()
df_map_src["Unidades_Pack"] = df_map_src["Tamanho"].apply(unidades_por_pack).astype(int)
df_map_src["Tipo_Pack"] = df_map_src["Tamanho"].apply(tipo_pack)
df_map_src["Volume_Origem"] = pd.to_numeric(df_map_src["Volume"], errors="coerce")
df_map_src["Volume_Tamanho"] = df_map_src["Tamanho"].apply(tamanho_para_volume_ml)
df_map_src["Volume"] = df_map_src.apply(volume_unitario_produto, axis=1)
df_map_src["Volume"] = normaliza_volume(df_map_src["Volume"])
 
mask_valid = (
    df_map_src["ID_Inventario"].ne("") &
    df_map_src["Marca"].ne("") &
    df_map_src["Descricao"].ne("") &
    df_map_src["Tamanho"].ne("")
 )
df_map_src = df_map_src[mask_valid].copy()
 
df_map_src["_nk"] = (
    df_map_src["Marca"].str.upper() + "|" +
    df_map_src["Descricao_Norm"] + "|" +
    df_map_src["Tamanho_Norm"] + "|" +
    volume_para_chave(df_map_src["Volume"]) + "|" +
    df_map_src["Unidades_Pack"].astype(str) + "|" +
    df_map_src["Tipo_Pack"]
)
 
prioridade_fonte_map = {"purchases": 1, "sales": 2, "end_inventory": 3, "beg_inventory": 4}
df_map_src["Prioridade_Fonte"] = df_map_src["Fonte"].map(prioridade_fonte_map).fillna(99)
df_map_src = (df_map_src.sort_values(["ID_Inventario", "Prioridade_Fonte"]).drop_duplicates(subset=["ID_Inventario"], keep="first"))
 
df_map = (df_map_src.merge(df_prod_nk, on="_nk", how="inner")[["ID_Inventario", "ID_Produto"]].drop_duplicates(subset=["ID_Inventario"], keep="first").reset_index(drop=True))
 
if len(df_map) > 0:
    try:
        upsert_sql = text("""
            INSERT INTO produto_inventario (ID_Inventario, ID_Produto)
            VALUES (:ID_Inventario, :ID_Produto)
            ON DUPLICATE KEY UPDATE ID_Produto = VALUES(ID_Produto)
        """)

        tamanho_lote = 50000
        total_lotes = (len(df_map) + tamanho_lote - 1) // tamanho_lote
        intervalo_progresso = 25

        for indice_lote in range(total_lotes):
            inicio_lote = indice_lote * tamanho_lote
            fim_lote = min((indice_lote + 1) * tamanho_lote, len(df_map))
            lote = df_map.iloc[inicio_lote:fim_lote].copy()

            lote = lote.astype(object).where(pd.notna(lote), None)

            registos_lote: list[dict[str, Any]] = [
                {str(chave): valor for chave, valor in linha.items()}
                for linha in lote.to_dict(orient="records")
            ]

            with engine.begin() as conn:
                conn.execute(upsert_sql, registos_lote)

            lote_atual = indice_lote + 1
            if lote_atual == 1 or lote_atual % intervalo_progresso == 0 or lote_atual == total_lotes:
                print(f'Progresso produto_inventario: lote {lote_atual}/{total_lotes} ({len(lote)} linhas)')

        print("Inserção concluída.")
    except Exception as e:
        print("Erro durante inserção em produto_inventario:", e)
        engine.dispose()
else:
    print("Nenhuma linha nova para inserir em produto_inventario.")


C:\Users\Gonçalo\AppData\Local\Temp\ipykernel_22584\2906687462.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_map_src = pd.concat([


Progresso produto_inventario: lote 1/6 (50000 linhas)
Progresso produto_inventario: lote 6/6 (26389 linhas)
Inserção concluída.


# Fact Table (Preço)

In [18]:
def normaliza_txt(serie):
    return (
        serie.astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )

df_beg = pd.read_sql(text("""
    SELECT
        InventoryId AS ID_Inventario,
        startDate AS Data,
        Price AS Preco_Produto
    FROM beg_inventory
    WHERE Price IS NOT NULL
"""), con=engine)

df_end = pd.read_sql(text("""
    SELECT
        InventoryId AS ID_Inventario,
        endDate AS Data,
        Price AS Preco_Produto
    FROM end_inventory
    WHERE Price IS NOT NULL
"""), con=engine)

df_preco = pd.concat([df_beg, df_end], ignore_index=True)
print('Linhas carregadas de beg_inventory e end_inventory:', len(df_preco))

df_preco['Preco_Produto'] = pd.to_numeric(df_preco['Preco_Produto'], errors='coerce')
df_preco['ID_Inventario'] = normaliza_txt(df_preco['ID_Inventario']).str.upper()
df_preco['Data'] = pd.to_datetime(df_preco['Data'], errors='coerce').dt.normalize()

df_fornecedor_produto = pd.read_sql(text("""
    SELECT
        InventoryId AS ID_Inventario,
        MIN(VendorNumber) AS ID_Fornecedor
    FROM purchases
    GROUP BY InventoryId
"""), con=engine)
df_fornecedor_produto['ID_Inventario'] = normaliza_txt(df_fornecedor_produto['ID_Inventario']).str.upper()
df_fornecedor_produto['ID_Fornecedor'] = pd.to_numeric(df_fornecedor_produto['ID_Fornecedor'], errors='coerce').astype('Int64')
df_preco = df_preco.merge(df_fornecedor_produto, on='ID_Inventario', how='left')

df_fornecedor = pd.read_sql(text('SELECT ID_Fornecedor FROM fornecedor'), con=engine)
df_fornecedor['ID_Fornecedor'] = pd.to_numeric(df_fornecedor['ID_Fornecedor'], errors='coerce').astype('Int64')
df_preco = df_preco.merge(df_fornecedor, on='ID_Fornecedor', how='inner')

df_cal = pd.read_sql(text('SELECT ID_Calendario, Data FROM calendario'), con=engine)
df_cal['Data'] = pd.to_datetime(df_cal['Data'], errors='coerce').dt.normalize()
df_preco = df_preco.merge(df_cal, on='Data', how='left')

df_prod_map = pd.read_sql(text('SELECT ID_Produto, ID_Inventario FROM produto_inventario'), con=engine)
df_prod_map['ID_Inventario'] = normaliza_txt(df_prod_map['ID_Inventario']).str.upper()
df_preco = df_preco.merge(
    df_prod_map.rename(columns={'ID_Produto': 'ID_Produto_ponte'}),
    on='ID_Inventario',
    how='left',
)
df_preco['ID_Produto'] = pd.to_numeric(df_preco['ID_Produto_ponte'], errors='coerce')

df_preco = df_preco[[
    'ID_Calendario',
    'ID_Produto',
    'ID_Fornecedor',
    'Preco_Produto',
]].copy()

df_preco = df_preco.dropna(subset=['ID_Calendario', 'ID_Produto', 'ID_Fornecedor', 'Preco_Produto'])
df_preco['ID_Calendario'] = pd.to_numeric(df_preco['ID_Calendario'], errors='coerce').astype(int)
df_preco['ID_Produto'] = pd.to_numeric(df_preco['ID_Produto'], errors='coerce').astype(int)
df_preco['ID_Fornecedor'] = pd.to_numeric(df_preco['ID_Fornecedor'], errors='coerce').astype(int)

df_preco = df_preco.drop_duplicates(subset=['ID_Calendario', 'ID_Produto', 'ID_Fornecedor'], keep='last')

existentes = pd.read_sql(
    text('SELECT ID_Calendario, ID_Produto, ID_Fornecedor FROM preco'),
    con=engine,
)
existentes['ID_Calendario'] = pd.to_numeric(existentes['ID_Calendario'], errors='coerce').astype('Int64')
existentes['ID_Produto'] = pd.to_numeric(existentes['ID_Produto'], errors='coerce').astype('Int64')
existentes['ID_Fornecedor'] = pd.to_numeric(existentes['ID_Fornecedor'], errors='coerce').astype('Int64')

df_preco = df_preco.merge(
    existentes.assign(_exists=1),
    on=['ID_Calendario', 'ID_Produto', 'ID_Fornecedor'],
    how='left',
)
df_preco = df_preco[df_preco['_exists'].isna()].drop(columns=['_exists'])

print('Linhas novas a inserir em preco:', len(df_preco))

if len(df_preco) > 0:
    try:
        tamanho_lote = 50000
        total_lotes = (len(df_preco) + tamanho_lote - 1) // tamanho_lote
        intervalo_progresso = 25

        for indice_lote in range(total_lotes):
            inicio_lote = indice_lote * tamanho_lote
            fim_lote = min((indice_lote + 1) * tamanho_lote, len(df_preco))
            lote = df_preco.iloc[inicio_lote:fim_lote]

            with engine.begin() as conn:
                lote.to_sql('preco', con=conn, if_exists='append', index=False)

            lote_atual = indice_lote + 1
            if lote_atual == 1 or lote_atual % intervalo_progresso == 0 or lote_atual == total_lotes:
                print(f'Progresso preco: lote {lote_atual}/{total_lotes} ({len(lote)} linhas)')

        print('Insercao concluida.')
    except Exception as e:
        print('Erro durante insercao em preco:', e)
        engine.dispose()
else:
    print('Nenhuma linha nova para inserir em preco.')

Linhas carregadas de beg_inventory e end_inventory: 431018
Linhas novas a inserir em preco: 15696
Progresso preco: lote 1/1 (15696 linhas)
Insercao concluida.


# Fact Table (Inventário)

In [19]:

df_beg_inventory = pd.read_sql(text("SELECT * FROM beg_inventory"), con=engine)
df_end_inventory = pd.read_sql(text("SELECT * FROM end_inventory"), con=engine)


print('Linhas fonte beg_inventory:', len(df_beg_inventory))
print('Linhas fonte end_inventory:', len(df_end_inventory))

qtd_col_beg = 'onHand'
qtd_col_end = 'onHand'

if qtd_col_beg not in df_beg_inventory.columns:
    raise ValueError(
        f"Coluna 'onHand' não encontrada em beg_inventory. "
        f"Colunas disponíveis: {df_beg_inventory.columns.tolist()}"
    )
if qtd_col_end not in df_end_inventory.columns:
    raise ValueError(
        f"Coluna 'onHand' não encontrada em end_inventory. "
        f"Colunas disponíveis: {df_end_inventory.columns.tolist()}"
    )

for df, table_name, date_col in [
    (df_beg_inventory, 'beg_inventory', 'startDate'),
    (df_end_inventory, 'end_inventory', 'endDate')
]:
    if 'InventoryId' not in df.columns:
        raise ValueError(f'InventoryId não encontrada em {table_name}')
    if 'Store' not in df.columns:
        raise ValueError(f'Store não encontrada em {table_name}')
    if date_col not in df.columns:
        raise ValueError(f'{date_col} não encontrada em {table_name}')

df_beg = df_beg_inventory.rename(columns={
    'InventoryId': 'ID_Inventario',
    'Store': 'N_Loja',
    'startDate': 'Data',
    qtd_col_beg: 'Existencias',
})[['ID_Inventario', 'N_Loja', 'Data', 'Existencias']].copy()

df_end = df_end_inventory.rename(columns={
    'InventoryId': 'ID_Inventario',
    'Store': 'N_Loja',
    'endDate': 'Data',
    qtd_col_end: 'Existencias',
})[['ID_Inventario', 'N_Loja', 'Data', 'Existencias']].copy()

for df in [df_beg, df_end]:
    df['Existencias'] = pd.to_numeric(df['Existencias'], errors='coerce')
    df['ID_Inventario'] = (
        df['ID_Inventario']
        .astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.upper()
    )
    df['N_Loja'] = (df['N_Loja'].astype('string').str.strip())
    df['StoreCodigo'] = (df['N_Loja'].str.split('_', n=1).str[0].str.strip())
    df['Data'] = pd.to_datetime(df['Data'], errors='coerce').dt.normalize()

df_inventario = pd.concat([df_beg, df_end], ignore_index=True)

mask_base_nulos = df_inventario[['ID_Inventario', 'StoreCodigo', 'Data', 'Existencias']].isna().any(axis=1)
df_inventario = df_inventario[~mask_base_nulos].copy()

df_produto = pd.read_sql(text('SELECT ID_Produto, ID_Inventario FROM produto_inventario'), con=engine)
df_produto['ID_Inventario'] = (
    df_produto['ID_Inventario']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.upper()
)
df_inventario = df_inventario.merge(df_produto, on='ID_Inventario', how='left')
sem_produto = int(df_inventario['ID_Produto'].isna().sum())

df_loja = pd.read_sql(text('SELECT ID_Loja, N_Loja FROM loja'), con=engine)
df_loja['StoreCodigo'] = (
    df_loja['N_Loja']
    .astype('string')
    .str.strip()
    .str.split('_', n=1).str[0]
    .str.strip()
)
df_inventario = df_inventario.merge(df_loja[['ID_Loja', 'StoreCodigo']], on='StoreCodigo', how='left')

df_cal = pd.read_sql(text('SELECT ID_Calendario, Data FROM calendario'), con=engine)
df_cal['Data'] = pd.to_datetime(df_cal['Data'], errors='coerce').dt.normalize()
df_inventario = df_inventario.merge(df_cal, on='Data', how='left')


df_inventario = df_inventario[[
    'ID_Calendario',
    'ID_Loja',
    'ID_Produto',
    'Existencias',
    'ID_Inventario',
]].copy()

cols = ['ID_Calendario', 'ID_Loja', 'ID_Produto', 'Existencias', 'ID_Inventario']

df_inventario[df_inventario[cols].isna().any(axis=1)].head(20)

mask_facto_nulos = df_inventario[['ID_Calendario', 'ID_Loja', 'ID_Produto', 'Existencias', 'ID_Inventario']].isna().any(axis=1)
df_inventario = df_inventario[~mask_facto_nulos].copy()

df_inventario['ID_Calendario'] = df_inventario['ID_Calendario'].astype(int)
df_inventario['ID_Loja'] = df_inventario['ID_Loja'].astype(int)
df_inventario['ID_Produto'] = df_inventario['ID_Produto'].astype(int)
df_inventario['Existencias'] = pd.to_numeric(df_inventario['Existencias'], errors='coerce').fillna(0).astype(int)
df_inventario['ID_Inventario'] = df_inventario['ID_Inventario'].astype('string').str.strip().str.upper()

df_inventario = df_inventario.drop_duplicates(subset=['ID_Calendario', 'ID_Loja', 'ID_Produto'], keep='last')


existente_inv = pd.read_sql(text('SELECT ID_Calendario, ID_Loja, ID_Produto FROM inventario'), con=engine)
df_inventario = df_inventario.merge(existente_inv.assign(_exists=1), on=['ID_Calendario', 'ID_Loja', 'ID_Produto'], how='left')

df_inventario = df_inventario[df_inventario['_exists'].isna()].drop(columns=['_exists'])

print('Linhas novas a inserir em inventario:', len(df_inventario))

if len(df_inventario) > 0:
    df_inventario.to_sql('inventario', con=engine, if_exists='append', index=False)
    print(f'Inseridas {len(df_inventario)} linhas em inventario.')
else:
    print('Nenhuma linha nova para inserir em inventario.')

Linhas fonte beg_inventory: 206529
Linhas fonte end_inventory: 224489
Linhas novas a inserir em inventario: 431018
Inseridas 431018 linhas em inventario.


# Fact Table (Venda)

In [20]:
df_sales_fact = pd.read_sql(text("SELECT InventoryId, Store, SalesDate, SalesPrice, SalesDollars, SalesQuantity, ExciseTax FROM sales"), con=engine)

for col in ['SalesPrice', 'SalesDollars', 'SalesQuantity', 'ExciseTax']:
    df_sales_fact[col] = pd.to_numeric(df_sales_fact[col], errors='coerce')

df_sales_fact['ID_Inventario'] = (
    df_sales_fact['InventoryId']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.upper()
)
df_sales_fact['N_Loja'] = (
    df_sales_fact['Store']
    .astype('string')
    .str.strip()
)
df_sales_fact['StoreCodigo'] = (
    df_sales_fact['N_Loja']
    .str.split('_', n=1).str[0]
    .str.strip()
)
df_sales_fact['Data'] = pd.to_datetime(df_sales_fact['SalesDate'], errors='coerce').dt.normalize()

df_produto = pd.read_sql(text('SELECT ID_Produto, ID_Inventario FROM produto_inventario'), con=engine)
df_produto['ID_Inventario'] = (
    df_produto['ID_Inventario']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.upper()
)
df_sales_fact = df_sales_fact.merge(df_produto, on='ID_Inventario', how='left')

df_loja = pd.read_sql(text('SELECT ID_Loja, N_Loja FROM loja'), con=engine)
df_loja['StoreCodigo'] = (
    df_loja['N_Loja']
    .astype('string')
    .str.strip()
    .str.split('_', n=1).str[0]
    .str.strip()
)
df_sales_fact = df_sales_fact.merge(df_loja[['ID_Loja', 'StoreCodigo']], on='StoreCodigo', how='left')

df_cal = pd.read_sql(text('SELECT ID_Calendario, Data FROM calendario'), con=engine)
df_cal['Data'] = pd.to_datetime(df_cal['Data'], errors='coerce').dt.normalize()
df_sales_fact = df_sales_fact.merge(df_cal, on='Data', how='left')

df_sales_fact = df_sales_fact[[
    'ID_Calendario',
    'ID_Loja',
    'ID_Produto',
    'ID_Inventario',
    'SalesPrice',
    'SalesDollars',
    'SalesQuantity',
    'ExciseTax',
]]

df_sales_fact = df_sales_fact.rename(columns={
    'SalesPrice': 'Preco_Produto_Venda',
    'SalesDollars': 'Valor_Venda',
    'SalesQuantity': 'Quantidade_Vendida',
    'ExciseTax': 'Imposto_Sobre_Venda',
})

df_sales_fact = df_sales_fact.dropna(subset=['ID_Calendario', 'ID_Loja', 'ID_Produto', 'ID_Inventario', 'Preco_Produto_Venda', 'Valor_Venda', 'Quantidade_Vendida', 'Imposto_Sobre_Venda'])

df_sales_fact['ID_Calendario'] = df_sales_fact['ID_Calendario'].astype(int)
df_sales_fact['ID_Loja'] = df_sales_fact['ID_Loja'].astype(int)
df_sales_fact['ID_Produto'] = df_sales_fact['ID_Produto'].astype(int)
df_sales_fact['ID_Inventario'] = df_sales_fact['ID_Inventario'].astype('string').str.strip().str.upper()

df_sales_fact = df_sales_fact.drop_duplicates(subset=['ID_Calendario', 'ID_Loja', 'ID_Produto'], keep='last')

existente_sales = pd.read_sql(text('SELECT ID_Calendario, ID_Loja, ID_Produto FROM venda'), con=engine)
df_sales_fact = df_sales_fact.merge(existente_sales.assign(_exists=1), on=['ID_Calendario', 'ID_Loja', 'ID_Produto'], how='left')
df_sales_fact = df_sales_fact[df_sales_fact['_exists'].isna()].drop(columns=['_exists'])

print('Linhas novas a inserir em venda:', len(df_sales_fact))

if len(df_sales_fact) > 0:
    try:
        tamanho_lote = 50000
        total_lotes = (len(df_sales_fact) + tamanho_lote - 1) // tamanho_lote
        intervalo_progresso = 25

        for indice_lote in range(total_lotes):
            inicio_lote = indice_lote * tamanho_lote
            fim_lote = min((indice_lote + 1) * tamanho_lote, len(df_sales_fact))
            lote = df_sales_fact.iloc[inicio_lote:fim_lote]

            with engine.begin() as conn:
                lote.to_sql('venda', con=conn, if_exists='append', index=False)
            lote_atual = indice_lote + 1
            if lote_atual == 1 or lote_atual % intervalo_progresso == 0 or lote_atual == total_lotes:
                print(f'Progresso venda: lote {lote_atual}/{total_lotes} ({len(lote)} linhas)')

        print('Inserção concluída.')
    except Exception as e:
        print('Erro durante inserção em venda:', e)
        engine.dispose()
else:
    print('Nenhuma linha nova para inserir em venda.')

Linhas novas a inserir em venda: 12825363
Progresso venda: lote 1/257 (50000 linhas)
Progresso venda: lote 25/257 (50000 linhas)
Progresso venda: lote 50/257 (50000 linhas)
Progresso venda: lote 75/257 (50000 linhas)
Progresso venda: lote 100/257 (50000 linhas)
Progresso venda: lote 125/257 (50000 linhas)
Progresso venda: lote 150/257 (50000 linhas)
Progresso venda: lote 175/257 (50000 linhas)
Progresso venda: lote 200/257 (50000 linhas)
Progresso venda: lote 225/257 (50000 linhas)
Progresso venda: lote 250/257 (50000 linhas)
Progresso venda: lote 257/257 (25363 linhas)
Inserção concluída.


# Fact Table (Compra)

In [4]:
df_compra_src = pd.read_sql(text("""
    WITH compras_por_fatura AS (
        SELECT
            PONumber,
            MIN(Store) AS N_Loja
        FROM purchases
        GROUP BY PONumber
    )
    SELECT
        i.VendorNumber AS ID_Fornecedor,
        i.PONumber     AS N_Fatura,
        c.N_Loja       AS N_Loja,
        i.InvoiceDate  AS InvoiceDate,
        i.Freight      AS Frete
    FROM invoice_purchases i
    LEFT JOIN compras_por_fatura c
        ON i.PONumber = c.PONumber
"""), con=engine)

print('Linhas fonte (invoice_purchases -> compra):', len(df_compra_src))

df_compra_src['ID_Fornecedor'] = pd.to_numeric(df_compra_src['ID_Fornecedor'], errors='coerce').astype('Int64')
df_compra_src['N_Fatura'] = pd.to_numeric(df_compra_src['N_Fatura'], errors='coerce').astype('Int64')
df_compra_src['N_Loja'] = df_compra_src['N_Loja'].astype('string').str.strip()
df_compra_src['StoreCodigo'] = (
    df_compra_src['N_Loja']
    .str.split('_', n=1).str[0]
    .str.strip()
)

df_compra_src['InvoiceDate'] = pd.to_datetime(df_compra_src['InvoiceDate'], errors='coerce').dt.normalize()
df_compra_src['Frete'] = pd.to_numeric(df_compra_src['Frete'], errors='coerce').fillna(0.0)

src_cols = [
    'ID_Fornecedor', 'N_Fatura', 'N_Loja', 'StoreCodigo',
    'InvoiceDate', 'Frete',
]
df_compra_src = df_compra_src.drop_duplicates(subset=src_cols, keep='first')
print('Linhas apos agrupamento por fatura:', len(df_compra_src))

df_fornecedor = pd.read_sql(text('SELECT ID_Fornecedor FROM fornecedor'), con=engine)
df_fornecedor['ID_Fornecedor'] = pd.to_numeric(df_fornecedor['ID_Fornecedor'], errors='coerce').astype('Int64')
df_compra_src = df_compra_src.merge(df_fornecedor, on='ID_Fornecedor', how='inner')

df_loja = pd.read_sql(text('SELECT ID_Loja, N_Loja FROM loja'), con=engine)
df_loja['N_Loja'] = df_loja['N_Loja'].astype('string').str.strip()
df_loja['StoreCodigo'] = (
    df_loja['N_Loja']
    .str.split('_', n=1).str[0]
    .str.strip()
)

df_cal = pd.read_sql(text('SELECT ID_Calendario, Data FROM calendario'), con=engine)
df_cal['Data'] = pd.to_datetime(df_cal['Data'], errors='coerce').dt.normalize()

df_compra_src = df_compra_src.rename(columns={'InvoiceDate': 'Data'})
df_compra = df_compra_src.merge(df_cal, on='Data', how='left')
df_compra = df_compra.merge(df_loja[['StoreCodigo', 'ID_Loja']], on='StoreCodigo', how='left')

df_compra = df_compra[[
    'ID_Calendario',
    'ID_Fornecedor',
    'N_Fatura',
    'ID_Loja',
    'Frete',
]].copy()

df_compra = df_compra.dropna(subset=[
    'ID_Calendario',
    'ID_Fornecedor',
    'N_Fatura',
    'ID_Loja',
    'Frete',
])

df_compra['ID_Calendario'] = df_compra['ID_Calendario'].astype(int)
df_compra['ID_Fornecedor'] = df_compra['ID_Fornecedor'].astype(int)
df_compra['N_Fatura'] = df_compra['N_Fatura'].astype(int)
df_compra['ID_Loja'] = df_compra['ID_Loja'].astype(int)
df_compra['Frete'] = df_compra['Frete'].astype(float)

compra_key_cols = [
    'ID_Calendario', 'ID_Fornecedor', 'ID_Loja', 'N_Fatura',
]
df_compra = df_compra.drop_duplicates(subset=compra_key_cols, keep='first')

existente = pd.read_sql(text("""
    SELECT
        ID_Calendario, ID_Fornecedor, N_Fatura, ID_Loja
    FROM compra
"""), con=engine)

if len(existente) > 0:
    existente['ID_Calendario'] = pd.to_numeric(existente['ID_Calendario'], errors='coerce').astype('Int64')
    existente['ID_Fornecedor'] = pd.to_numeric(existente['ID_Fornecedor'], errors='coerce').astype('Int64')
    existente['N_Fatura'] = pd.to_numeric(existente['N_Fatura'], errors='coerce').astype('Int64')
    existente['ID_Loja'] = pd.to_numeric(existente['ID_Loja'], errors='coerce').astype('Int64')

    match_cols = compra_key_cols
    df_compra = df_compra.merge(existente.assign(_exists=1), on=match_cols, how='left')
    df_compra = df_compra[df_compra['_exists'].isna()].drop(columns=['_exists'])

print('Linhas novas a inserir em compra:', len(df_compra))

if len(df_compra) > 0:
    try:
        tamanho_lote = 50000
        total_lotes = (len(df_compra) + tamanho_lote - 1) // tamanho_lote
        intervalo_progresso = 25

        for indice_lote in range(total_lotes):
            inicio_lote = indice_lote * tamanho_lote
            fim_lote = min((indice_lote + 1) * tamanho_lote, len(df_compra))
            lote = df_compra.iloc[inicio_lote:fim_lote]

            with engine.begin() as conn:
                lote.to_sql('compra', con=conn, if_exists='append', index=False)

            lote_atual = indice_lote + 1
            if lote_atual == 1 or lote_atual % intervalo_progresso == 0 or lote_atual == total_lotes:
                print(f'Progresso compra: lote {lote_atual}/{total_lotes} ({len(lote)} linhas)')

        print('Insercao concluida.')
    except Exception as e:
        print('Erro durante insercao em compra:', e)
        engine.dispose()
else:
    print('Nenhuma linha nova para inserir em compra.')

Linhas fonte (invoice_purchases -> compra): 5543
Linhas apos agrupamento por fatura: 5543
Linhas novas a inserir em compra: 5543
Progresso compra: lote 1/1 (5543 linhas)
Insercao concluida.


# Fact Table (Detalhe Compra)

In [22]:
df_src = pd.read_sql(text("SELECT * FROM purchases"), con=engine)

print('Linhas fonte:', len(df_src))

df_src = df_src.drop_duplicates(keep='first')

df_src = df_src.rename(columns={
    'InvoiceDate': 'Data',
    'VendorNumber': 'ID_Fornecedor',
    'InventoryId': 'ID_Inventario',
    'Store': 'N_Loja',
    'PONumber': 'N_Fatura',
    'Quantity': 'Quantidade_Comprada',
    'PurchasePrice': 'Preco_Compra_Produto',
    'Dollars': 'Valor_Linha_Compra',
})

df_src['Data'] = pd.to_datetime(df_src['Data'], errors='coerce').dt.normalize()
df_src['ID_Fornecedor'] = pd.to_numeric(df_src['ID_Fornecedor'], errors='coerce').astype('Int64')
df_src['N_Fatura'] = pd.to_numeric(df_src['N_Fatura'], errors='coerce').astype('Int64')
df_src['Quantidade_Comprada'] = pd.to_numeric(df_src['Quantidade_Comprada'], errors='coerce')
df_src['Preco_Compra_Produto'] = pd.to_numeric(df_src['Preco_Compra_Produto'], errors='coerce')
df_src['Valor_Linha_Compra'] = pd.to_numeric(df_src['Valor_Linha_Compra'], errors='coerce')
df_src['ID_Inventario'] = (
    df_src['ID_Inventario']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.upper()
)
df_src['N_Loja'] = df_src['N_Loja'].astype('string').str.strip()
df_src['StoreCodigo'] = (
    df_src['N_Loja']
    .str.split('_', n=1).str[0]
    .str.strip()
)

df_produto = pd.read_sql(text('SELECT ID_Produto, ID_Inventario FROM produto_inventario'), con=engine)
df_produto['ID_Inventario'] = (
    df_produto['ID_Inventario']
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.upper()
)

df_loja = pd.read_sql(text('SELECT ID_Loja, N_Loja FROM loja'), con=engine)
df_loja['N_Loja'] = df_loja['N_Loja'].astype('string').str.strip()
df_loja['StoreCodigo'] = (
    df_loja['N_Loja']
    .str.split('_', n=1).str[0]
    .str.strip()
)

df_fornecedor = pd.read_sql(text('SELECT ID_Fornecedor FROM fornecedor'), con=engine)
df_fornecedor['ID_Fornecedor'] = pd.to_numeric(df_fornecedor['ID_Fornecedor'], errors='coerce').astype('Int64')

df_cal = pd.read_sql(text('SELECT ID_Calendario, Data FROM calendario'), con=engine)
df_cal['Data'] = pd.to_datetime(df_cal['Data'], errors='coerce').dt.normalize()

df = df_src.merge(df_fornecedor, on='ID_Fornecedor', how='inner')

df = (
    df
    .merge(df_produto, on='ID_Inventario', how='left')
    .merge(df_loja[['ID_Loja', 'StoreCodigo']], on='StoreCodigo', how='left')
    .merge(df_cal, on='Data', how='left')
)

df = df[[
    'ID_Calendario',
    'ID_Fornecedor',
    'ID_Produto',
    'ID_Loja',
    'N_Fatura',
    'ID_Inventario',
    'Quantidade_Comprada',
    'Preco_Compra_Produto',
    'Valor_Linha_Compra',
]].copy()

df = df.dropna(subset=[
    'ID_Calendario',
    'ID_Fornecedor',
    'ID_Produto',
    'ID_Loja',
    'N_Fatura',
    'ID_Inventario',
    'Quantidade_Comprada',
    'Preco_Compra_Produto',
    'Valor_Linha_Compra',
])

df['ID_Calendario'] = df['ID_Calendario'].astype(int)
df['ID_Fornecedor'] = df['ID_Fornecedor'].astype(int)
df['ID_Produto'] = df['ID_Produto'].astype(int)
df['ID_Loja'] = df['ID_Loja'].astype(int)
df['N_Fatura'] = df['N_Fatura'].astype(int)
df['ID_Inventario'] = df['ID_Inventario'].astype('string').str.strip().str.upper()
df['Quantidade_Comprada'] = df['Quantidade_Comprada'].astype(int)
df['Preco_Compra_Produto'] = df['Preco_Compra_Produto'].astype(float)
df['Valor_Linha_Compra'] = df['Valor_Linha_Compra'].astype(float)

existente = pd.read_sql(text("""
    SELECT
        ID_Calendario, ID_Fornecedor, ID_Produto, ID_Loja,
        N_Fatura, ID_Inventario, Quantidade_Comprada,
        Preco_Compra_Produto, Valor_Linha_Compra
    FROM detalhe_compra
"""), con=engine)

if len(existente) > 0:
    existente['ID_Calendario'] = pd.to_numeric(existente['ID_Calendario'], errors='coerce').astype('Int64')
    existente['ID_Fornecedor'] = pd.to_numeric(existente['ID_Fornecedor'], errors='coerce').astype('Int64')
    existente['ID_Produto'] = pd.to_numeric(existente['ID_Produto'], errors='coerce').astype('Int64')
    existente['ID_Loja'] = pd.to_numeric(existente['ID_Loja'], errors='coerce').astype('Int64')
    existente['N_Fatura'] = pd.to_numeric(existente['N_Fatura'], errors='coerce').astype('Int64')
    existente['ID_Inventario'] = (
        existente['ID_Inventario']
        .astype('string')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
        .str.upper()
)
    existente['Quantidade_Comprada'] = pd.to_numeric(existente['Quantidade_Comprada'], errors='coerce').astype('Int64')
    existente['Preco_Compra_Produto'] = pd.to_numeric(existente['Preco_Compra_Produto'], errors='coerce').astype(float)
    existente['Valor_Linha_Compra'] = pd.to_numeric(existente['Valor_Linha_Compra'], errors='coerce').astype(float)

    match_cols = [
        'ID_Calendario', 'ID_Fornecedor', 'ID_Produto', 'ID_Loja',
        'N_Fatura', 'ID_Inventario', 'Quantidade_Comprada',
        'Preco_Compra_Produto', 'Valor_Linha_Compra',
    ]
    df = df.merge(existente.assign(_exists=1), on=match_cols, how='left')
    df = df[df['_exists'].isna()].drop(columns=['_exists'])

print('Linhas novas a inserir em detalhe_compra:', len(df))

if len(df) > 0:
    try:
        tamanho_lote = 50000
        total_lotes = (len(df) + tamanho_lote - 1) // tamanho_lote
        intervalo_progresso = 25

        for indice_lote in range(total_lotes):
            inicio_lote = indice_lote * tamanho_lote
            fim_lote = min((indice_lote + 1) * tamanho_lote, len(df))
            lote = df.iloc[inicio_lote:fim_lote]

            with engine.begin() as conn:
                lote.to_sql('detalhe_compra', con=conn, if_exists='append', index=False)
        
            lote_atual = indice_lote + 1
            if lote_atual == 1 or lote_atual % intervalo_progresso == 0 or lote_atual == total_lotes:
                print(f'Progresso detalhe_compra: lote {lote_atual}/{total_lotes} ({len(lote)} linhas)')

        print('Inserção concluída.')
    except Exception as e:
        print('Erro durante inserção em detalhe_compra:', e)
        engine.dispose()
else:
    print('Nenhuma linha nova para inserir em detalhe_compra.')

Linhas fonte: 2372474
Linhas novas a inserir em detalhe_compra: 2372474
Progresso detalhe_compra: lote 1/48 (50000 linhas)
Progresso detalhe_compra: lote 25/48 (50000 linhas)
Progresso detalhe_compra: lote 48/48 (22474 linhas)
Inserção concluída.
